In [3]:
import re
import pandas as pd

# === 1. Read the old TAP result ===
# Suppose you exported the old result as a CSV with headers: dp_id, obs_creator_did
old_df = pd.read_csv("./data/sphere_tap_results.csv")

# Extract FITS filenames from the ivo:// URLs
old_df["fits_name"] = old_df["obs_creator_did"].apply(
    lambda x: re.search(r"OB_\d+_\d{4}-\d{2}-\d{2}T\d{2}-\d{2}\.fits", x).group(0)
    if isinstance(x, str) and re.search(r"OB_\d+_\d{4}-\d{2}-\d{2}T\d{2}-\d{2}\.fits", x)
    else None
)

old_fits = set(old_df["fits_name"].dropna())

# === 2. Read the new data file list ===
# Suppose you have it as a simple text file, one FITS per line
with open("./data/batch_25400_filelist.txt") as f:
    new_fits = {line.strip() for line in f if line.strip()}

# === 3. Compare ===
common = new_fits & old_fits
new_only = new_fits - old_fits
old_only = old_fits - new_fits

# === 4. Report ===
print(f"Total old entries: {len(old_fits)}")
print(f"Total new entries: {len(new_fits)}")
print(f"Common FITS files: {len(common)}")
print(f"New (not in old): {len(new_only)}")
print(f"Old (missing in new): {len(old_only)}")

# List up to 5 “old (missing in new)” FITS names
print("\nOld (missing in new) — first 5 entries:")
for name in list(sorted(old_only))[:5]:
    print("  ", name)

# # Optional: Save differences to files
# pd.Series(sorted(new_only)).to_csv("./data/new_unique.csv", index=False, header=["fits_name"])
# pd.Series(sorted(old_only)).to_csv("./data/old_unique.csv", index=False, header=["fits_name"])

Total old entries: 602
Total new entries: 3265
Common FITS files: 597
New (not in old): 2668
Old (missing in new): 5

Old (missing in new) — first 5 entries:
   OB_2233863_2019-06-14T06-28.fits
   OB_2437396_2019-05-27T23-40.fits
   OB_2437399_2019-05-28T00-59.fits
   OB_2533861_2019-10-26T07-42.fits
   OB_2562987_2019-08-15T07-30.fits
